# Lesson 3 · Circular motion and gravitational orbits

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/03_circular_motion_and_orbits.ipynb)

**For:** A-level Physics (for example AQA 3.6.1.1 and 3.7.2, OCR A 5.2 and 5.4, Edexcel Topics 6 and 12) and IB Physics (A.2 circular motion and D.1 gravitational fields). Check your own specification for the exact wording.

**Before this lesson:** Lesson 2, *Projectile motion* (splitting motion into perpendicular components). Also assumed: Newton's laws and kinetic and potential energy.

**Leads to:** Lesson 5, *Electric fields and potential*; Lesson 6, *Charged particles in electric and magnetic fields*; Lesson 9, *Ideal gases and kinetic theory*.

**You will be able to:**

1. use $\omega = 2\pi/T$, $v = \omega r$ and $a = v^2/r = \omega^2 r$, and explain why circular motion needs a resultant force
2. apply Newton's law of gravitation to circular orbits: $v = \sqrt{GM/r}$ and $T^2 = 4\pi^2 r^3/GM$
3. explain what is special about a geostationary orbit
4. use Kepler's third law, and test a power law with a log-log graph
5. describe the energy of an orbit and calculate escape velocity $v_{esc} = \sqrt{2GM/r}$

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

In [ ]:
# Setup: installs the qplay package if it is missing (for example on Google Colab).
import importlib.util, subprocess, sys
if importlib.util.find_spec("qplay") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "quantum-playground"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import mechanics

G = 6.674e-11          # N m^2 kg^-2
M_EARTH = 5.972e24     # kg
R_EARTH = 6.371e6      # m


def orbit(speed, t_max, n=4000):
    """A light planet starting at (1, 0) moving in +y with the given speed, round a star of mass 1 (G = 1).

    The planet is a billion times lighter than the star, so the star hardly moves.
    Returns (t, x, y, vx, vy) for the planet.
    """
    t, pos, vel = mechanics.nbody([1.0, 1e-9], [[0, 0], [1, 0]], [[0, 0], [0, speed]], t_max, n=n)
    return t, pos[:, 1, 0], pos[:, 1, 1], vel[:, 1, 0], vel[:, 1, 1]


def measured_period(t, y):
    """Time between the first two upward crossings of the x axis."""
    up = np.where((y[:-1] < 0) & (y[1:] >= 0))[0]
    crossings = t[up] - y[up] * (t[up + 1] - t[up]) / (y[up + 1] - y[up])   # linear interpolation
    return crossings[1] - crossings[0] if len(crossings) > 1 else np.nan

## 1. Moving in a circle means accelerating

An object moving in a circle at constant speed is **accelerating**, because the *direction* of its velocity keeps changing. The acceleration points to the centre and has size

$$a = \frac{v^2}{r} = \omega^2 r,\qquad v = \omega r,\qquad \omega = \frac{2\pi}{T}.$$

By Newton's second law there must be a resultant force towards the centre, the **centripetal force** $F = \dfrac{mv^2}{r}$. It is not a new kind of force: it is whatever provides the pull (tension, friction, gravity...).

Below, gravity keeps a planet in a circle. The computer only knows Newton's law of gravitation; we then measure the acceleration from the motion and compare it with $v^2/r$.

In [ ]:
t, x, y, vx, vy = orbit(speed=1.0, t_max=2 * np.pi, n=2000)     # speed 1 at r = 1 is the circular speed when GM = 1
r = np.hypot(x, y)
v = np.hypot(vx, vy)
ax_num, ay_num = np.gradient(vx, t), np.gradient(vy, t)          # acceleration measured from the velocity
a = np.hypot(ax_num, ay_num)
towards_centre = -(ax_num * x + ay_num * y) / (a * r)            # cosine of the angle to the centre

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(x, y); ax1.plot(0, 0, "o", color="C1", ms=10)
step = 125
ax1.quiver(x[::step], y[::step], ax_num[::step], ay_num[::step], color="C3", scale=8, label="acceleration")
ax1.set_aspect("equal"); ax1.legend(loc="upper right"); ax1.set_title("Acceleration always points at the star")
ax2.plot(t, a, label="measured |a|"); ax2.plot(t, v**2 / r, "k--", label=r"$v^2/r$")
ax2.set_ylim(0, 1.5); ax2.set_xlabel("t"); ax2.legend(); ax2.set_title("Size of the acceleration")
print(f"speed stays between {v.min():.6f} and {v.max():.6f};  cos(angle to centre) >= {towards_centre[5:-5].min():.6f}")

The speed never changes, yet the acceleration is never zero: it is exactly $v^2/r$ and always aimed at the centre (the cosine of the angle is 1).

## 2. Gravity provides the centripetal force

For a satellite of mass $m$ in a circular orbit of radius $r$ around a planet of mass $M$, gravity is the centripetal force:

$$\frac{GMm}{r^2} = \frac{mv^2}{r}\quad\Rightarrow\quad v = \sqrt{\frac{GM}{r}},\qquad T = \frac{2\pi r}{v} = 2\pi\sqrt{\frac{r^3}{GM}} .$$

The satellite's own mass cancels: a spanner and the space station orbit together. The field strength at distance $r$ is $g = \dfrac{GM}{r^2}$, which is $9.81\ \mathrm{N\,kg^{-1}}$ at the Earth's surface.

In [ ]:
orbits = {"International Space Station": 420e3, "GPS satellite": 20_200e3, "geostationary": 35_786e3, "the Moon": 384_400e3 - R_EARTH}
print(f"{'':28s} {'height / km':>12s} {'g / N kg^-1':>12s} {'speed / km s^-1':>16s} {'period':>14s}")
for name, h in orbits.items():
    r = R_EARTH + h
    v = np.sqrt(G * M_EARTH / r)
    T = 2 * np.pi * np.sqrt(r**3 / (G * M_EARTH))
    period = f"{T / 60:.1f} min" if T < 2 * 3600 else f"{T / 3600:.2f} h" if T < 2 * 86400 else f"{T / 86400:.1f} days"
    print(f"{name:28s} {h / 1e3:12.0f} {G * M_EARTH / r**2:12.3f} {v / 1e3:16.2f} {period:>14s}")

**Exam tip:** a **geostationary** satellite orbits above the equator, in the same direction as the Earth spins, with a period of one day, so it stays above the same point. Every geostationary satellite has the same radius, about 42 000 km from the Earth's centre: the period fixes $r$.

(The Moon's period comes out as 27.5 days against the measured 27.3 days. The formula treats the Moon as light, but it is 1.2% of the Earth's mass; using $T^2 = 4\pi^2 r^3/G(M+m)$ closes the gap.)

## 3. Kepler's third law

Squaring the period formula gives **Kepler's third law**: $T^2 = \dfrac{4\pi^2}{GM}\,r^3$, so $T^2 \propto r^3$ for everything orbiting the same body. Kepler found it in 1619 from planetary data, decades before Newton explained it.

It also holds for **elliptical** orbits if $r$ is replaced by the semi-major axis $a$ (half the longest diameter). Below, the planet is launched at different speeds. Each one gives a different ellipse; we measure its period from the motion and compare it with $T = 2\pi a^{3/2}$ (in units where $GM = 1$).

**Predict (P1).** The planet launched at speed 1.2 follows an ellipse whose semi-major axis is about 1.8 times the radius of the circular orbit. Predict its period as a multiple of the circular orbit's period. Write your prediction down before you run the next cell.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
rows = []
for i, speed in enumerate([0.8, 0.9, 1.0, 1.1, 1.2, 1.3]):          # <- Try it: add other speeds below 1.41
    energy = 0.5 * speed**2 - 1.0                                   # per unit mass, GM = 1, r = 1
    a = -1 / (2 * energy)                                           # semi-major axis from the energy
    T_kepler = 2 * np.pi * a**1.5
    t, x, y, _, _ = orbit(speed, t_max=2.2 * T_kepler)
    rows.append((speed, a, measured_period(t, y), T_kepler))
    ax1.plot(x, y, color=f"C{i}", label=f"v = {speed}")
ax1.plot(0, 0, "*", color="orange", ms=14); ax1.set_aspect("equal"); ax1.legend(fontsize=8); ax1.set_title("Same start, different speeds")

for speed, a, T_meas, T_kep in rows:
    print(f"v = {speed:.1f}:  a = {a:.3f},  measured T = {T_meas:.4f},  Kepler 2*pi*a^1.5 = {T_kep:.4f}")

# Real planets: semi-major axis in AU and period in years (measured data)
planets = {"Mercury": (0.387, 0.241), "Venus": (0.723, 0.615), "Earth": (1.000, 1.000),
           "Mars": (1.524, 1.881), "Jupiter": (5.203, 11.86), "Saturn": (9.537, 29.46)}
a_p, T_p = np.array(list(planets.values())).T
ax2.loglog(a_p, T_p, "o")
for name, (ai, Ti) in planets.items():
    ax2.annotate(name, (ai, Ti), xytext=(5, -10), textcoords="offset points", fontsize=8)
slope = np.polyfit(np.log(a_p), np.log(T_p), 1)[0]
aa = np.logspace(-0.5, 1.1, 50); ax2.loglog(aa, aa**1.5, "k--", lw=1, label=r"$T = a^{3/2}$")
ax2.set_xlabel("semi-major axis / AU"); ax2.set_ylabel("period / years"); ax2.legend()
ax2.set_title(f"The planets: log-log slope = {slope:.4f}");

**Explain (P1).** Use Kepler's third law to explain the number you see.

On a log-log graph $T = k\,a^{3/2}$ becomes a straight line of gradient $3/2$, because $\log T = \tfrac32 \log a + \log k$. The planets give 1.500 to four figures. A log-log plot is the standard way to find an unknown power law from data.

## 4. Energy and escape

For a satellite in a circular orbit:

$$E_k = \tfrac12 mv^2 = \frac{GMm}{2r},\qquad E_p = -\frac{GMm}{r},\qquad E_{total} = -\frac{GMm}{2r}.$$

The total energy is **negative**: the satellite is bound. Give it enough kinetic energy to make the total zero and it escapes:

$$\tfrac12 m v_{esc}^2 - \frac{GMm}{r} = 0 \quad\Rightarrow\quad v_{esc} = \sqrt{\frac{2GM}{r}} = \sqrt2\, v_{circ}.$$

**Predict (P2).** The planet is launched at 0.7, 1.0, 1.3, 1.41, 1.414 ($\sqrt2$) and 1.6 times the circular speed. Which of these escape? Write your prediction down before you run the next cell.

In [ ]:
plt.figure(figsize=(6.5, 5.5))
for i, factor in enumerate([0.7, 1.0, 1.3, 1.41, np.sqrt(2), 1.6]):       # launch speed / circular speed
    t, x, y, vx, vy = orbit(factor, t_max=40, n=8000)
    energy = 0.5 * (vx**2 + vy**2) - 1 / np.hypot(x, y)
    keep = np.hypot(x, y) < 12
    plt.plot(x[keep], y[keep], color=f"C{i}", label=f"{factor:.3f} v_circ,  E = {energy[0]:+.3f}")
plt.plot(0, 0, "*", color="orange", ms=14)
plt.xlim(-8, 6); plt.ylim(-6, 8); plt.gca().set_aspect("equal"); plt.legend(fontsize=8, loc="upper left")
plt.title(r"Bound ($E<0$), escaping ($E \geq 0$)");

**Explain (P2).** What does the sign of the total energy $E$ in the legend tell you, and why are 1.41 and 1.414 so different?

At $1.41\,v_{circ}$ the ellipse is enormous but still closed; at exactly $\sqrt2\,v_{circ}$ the energy is zero and the path is a parabola that never returns. Above that it is a hyperbola.

**Exam tip:** escape velocity does not depend on the direction of launch (ignoring the atmosphere), only on the energy. And nothing is "escaping gravity": the field never becomes zero, the object simply never stops moving away.

---

## Exercises

Take $G = 6.67\times10^{-11}\ \mathrm{N\,m^2\,kg^{-2}}$, Earth's mass $5.97\times10^{24}$ kg and radius $6.37\times10^{6}$ m (these are the constants defined at the top). Give answers to 3 significant figures.

**Q1.** A 1200 kg car goes round a bend of radius 50 m at 15 m s$^{-1}$. Find (a) its centripetal acceleration, (b) the friction force needed.

**Q2.** A geostationary satellite has a period of 86 164 s (one sidereal day). Find (a) its orbital radius, (b) its height above the Earth's surface.

**Q3.** The International Space Station orbits 420 km above the surface. Find (a) its orbital speed, (b) its period in minutes.

**Q4.** Find the escape velocity from the Earth's surface.

**Q5.** The Moon orbits the Earth at a radius of $3.84\times10^{8}$ m with a period of 27.3 days. Use this to calculate the mass of the Earth.

**Q6 (simulation).** In the units of section 3 ($GM = 1$, start at $r = 1$), a planet is launched sideways at speed 1.2. Use `orbit` and `measured_period` to find its period.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_GM = G * M_EARTH
_r_geo = (_GM * 86164**2 / (4 * np.pi**2)) ** (1 / 3)
_r_iss = R_EARTH + 420e3
_t6, _, _y6, _, _ = orbit(1.2, t_max=40, n=8000)
_answers = {
    "Q1a": 15**2 / 50, "Q1b": 1200 * 15**2 / 50,
    "Q2a": _r_geo, "Q2b": _r_geo - R_EARTH,
    "Q3a": np.sqrt(_GM / _r_iss), "Q3b": 2 * np.pi * _r_iss / np.sqrt(_GM / _r_iss) / 60,
    "Q4": np.sqrt(2 * _GM / R_EARTH),
    "Q5": 4 * np.pi**2 * 3.84e8**3 / (G * (27.3 * 86400) ** 2),
    "Q6": measured_period(_t6, _y6),
}

# Replace None with your answers, in SI units (Q3b in minutes, Q6 in simulation units).
mine = {"Q1a": None, "Q1b": None, "Q2a": None, "Q2b": None, "Q3a": None, "Q3b": None, "Q4": None, "Q5": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** (a) $a = v^2/r = 15^2/50 = 4.50\ \mathrm{m\,s^{-2}}$ &nbsp; (b) $F = ma = 1200 \times 4.50 = 5400\ \mathrm{N}$, provided by friction between the tyres and the road.

**Q2.** (a) $T = 2\pi\sqrt{r^3/GM} \Rightarrow r = \left(\dfrac{GMT^2}{4\pi^2}\right)^{1/3} = \left(\dfrac{6.67\times10^{-11}\times5.97\times10^{24}\times86164^2}{4\pi^2}\right)^{1/3} = 4.22\times10^{7}\ \mathrm{m}$
(b) $h = r - R = 4.216\times10^7 - 6.37\times10^6 = 3.58\times10^{7}\ \mathrm{m}$, about 35 800 km.

**Q3.** $r = 6.371\times10^6 + 4.20\times10^5 = 6.791\times10^6$ m. (a) $v = \sqrt{GM/r} = 7.66\times10^{3}\ \mathrm{m\,s^{-1}}$ &nbsp; (b) $T = 2\pi r/v = 5570\ \mathrm{s} = 92.8$ minutes.

**Q4.** $v_{esc} = \sqrt{2GM/R} = \sqrt{2 \times 6.67\times10^{-11} \times 5.97\times10^{24} / 6.37\times10^6} = 1.12\times10^{4}\ \mathrm{m\,s^{-1}}$, about 11.2 km s$^{-1}$.

**Q5.** Rearranging Kepler's third law, $M = \dfrac{4\pi^2 r^3}{GT^2} = \dfrac{4\pi^2 \times (3.84\times10^8)^3}{6.67\times10^{-11} \times (27.3 \times 86400)^2} = 6.02\times10^{24}\ \mathrm{kg}$. Remember to convert days to seconds.

**Q6.** The energy per unit mass is $\tfrac12(1.2)^2 - 1 = -0.28$, so $a = -1/(2E) = 1.786$ and Kepler predicts $T = 2\pi a^{3/2} = 15.0$. The simulation agrees: see below.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}")

**Next:** in section 3 add the speed 1.4. The orbit is so stretched that you need a longer `t_max`. How does the measured period compare with Kepler's law?

### Predict and explain: answers

**P1.** About 2.4 times. $T \propto a^{3/2}$, so $T/T_{circ} = 1.786^{3/2} = 2.39$, and the printed periods give $14.99/6.283 = 2.39$. Making an orbit twice as big makes its period $2^{3/2} \approx 2.8$ times longer, not twice as long.

**P2.** Only $\sqrt2$ (just) and 1.6. Escape needs $E \geq 0$. At $1.41\,v_{circ}$ the energy is $-0.006$: still bound, on an enormous ellipse (semi-major axis about 84) that does come back, eventually. At exactly $\sqrt2$ the energy is zero and the path is a parabola that never returns.